# 00 · The six things a notebook package can do

By the end of this notebook you will have written, from nothing, the small set
of functions that every AI-in-Jupyter tool is made of. Every package in this
series — and the real `socratic-watchdog` running on the course server — is
those functions in a different order.

You are not going to type most of the code. You are going to write a **spec**
and a **failing test**, hand both to your AI, and check what comes back. That
is the skill this course is actually teaching, and it is why the functions here
are small: a 12-billion-parameter model cannot build a package from a
paragraph, but it can write `get_current_cell()` from a docstring and one
assertion, every time.

## The gap that makes any of this hard

Run a cell and you feel like the notebook is one program. It is two.

```
   your browser                 the kernel
   ┌────────────────┐          ┌────────────────┐
   │ the document   │  ─────>  │ python process │
   │ cells, outputs │  <─────  │ your variables │
   └────────────────┘          └────────────────┘
        knows                       knows
     everything                  almost nothing
   about the notebook            about the notebook
```

The kernel receives a string of code and sends back a result. It is never told
what cell that string came from, what is above it, or that a notebook exists at
all. Every interesting thing an AI notebook tool does starts by closing that
gap — and closing it is most of the work. The model call at the end is four
lines.

## The four layers

| | | |
|---|---|---|
| **L0** | read | what is in the notebook |
| **L1** | write | change what the notebook shows |
| **L2** | model | ask an LLM something |
| **L3** | trigger | run without being called |

This notebook builds L0 and L1. Notebook 01 adds L2, notebook 03 adds L3, and
from there you are assembling rather than building.

> **How to work through this page.** Each function gets four cells: a **spec**
> you read, a **test** you run and watch fail, a **build** cell you hand to your
> AI, and the test again. Do not skip watching the test fail — a test you never
> saw fail is a test you do not know works.

In [ ]:
def check(label):
    """Run a test and print one line. Use it as a decorator.

    @check("my function does the thing")
    def _():
        assert my_function(1) == 2

    A failing check prints the reason instead of stopping the notebook, so you
    can run the whole page and see everything that is still red.
    """
    def run(test):
        try:
            test()
            print(f"PASS  {label}")
        except AssertionError as e:
            print(f"FAIL  {label}" + (f"  ->  {e}" if str(e) else ""))
        except Exception as e:
            print(f"ERROR {label}  ->  {type(e).__name__}: {e}")
    return run


print("check() ready")

---
## Getting hold of the kernel

Everything starts with one object. `get_ipython()` returns the live shell — the
thing that ran this cell. Outside a notebook it returns `None`, and that
matters more than it sounds: your package will be imported by test runners and
by plain `python`, and it must not explode there.

In [ ]:
def shell():
    """The live IPython shell, or None when there is no kernel."""
    try:
        from IPython import get_ipython
        return get_ipython()
    except Exception:
        return None


print(type(shell()).__name__)

`ZMQInteractiveShell` means you are in a notebook kernel. `TerminalInteractive
Shell` means IPython in a terminal. `NoneType` means plain Python.

Have a look at what it carries. This is the whole map — everything in L0 and L1
comes out of one of these.

In [ ]:
ip = shell()

print("user_ns          — every variable you have defined:",
      len([k for k in ip.user_ns if not k.startswith("_")]), "public names")
print("user_ns['_ih']   — the source of every cell you have run:",
      len(ip.user_ns["_ih"]), "entries")
print("user_ns['_']     — the last output value:", repr(ip.user_ns.get("_"))[:40])
print("events           — hooks you can attach:", list(ip.events.callbacks)[:3], "...")
print("set_next_input   — ask the browser to write a cell:", callable(ip.set_next_input))

---
# L0 — reading

## 1. `strip_magics(source)`

The first function, and it exists because of a disagreement.

When you write `%%explain` at the top of a cell, IPython hands your magic the
cell source **with that line already removed**. But the notebook file on disk
stores the full text, magic line and all. So the same cell has two different
sources depending on who you ask, and any attempt to match one against the
other fails.

`strip_magics` normalises them. It drops the leading lines that start with `%`
(a magic) or `!` (a shell command) — and only the *leading* ones. A `%` in the
middle of real code is a modulo operator, not a magic, and eating it would
corrupt the student's program.

In [ ]:
@check("removes a cell magic")
def _():
    assert strip_magics("%%explain\nx = 1") == "x = 1"

@check("removes several leading magics")
def _():
    assert strip_magics("%load_ext nbkit\n%%explain\nx = 1") == "x = 1"

@check("removes a shell escape")
def _():
    assert strip_magics("!pip install nbkit\nimport nbkit") == "import nbkit"

@check("leaves ordinary code alone")
def _():
    assert strip_magics("x = 1") == "x = 1"

@check("does NOT touch a % inside real code")
def _():
    assert strip_magics("x = 10 % 3\ny = 2") == "x = 10 % 3\ny = 2"

@check("survives an empty cell")
def _():
    assert strip_magics("") == ""

Six red lines. Good — now you know what the tests actually check.

**Give your AI this:**

> Write a Python function `strip_magics(source: str) -> str`. It removes the
> leading lines of `source` that start with `%` or `!` (ignoring indentation),
> stops at the first line that does not, and returns the rest with surrounding
> whitespace stripped. A `%` that appears anywhere other than the start of a
> leading line must be left alone. Use only the standard library.

Notice what that prompt contains: the exact signature, the exact rule, and the
one edge case that is easy to get wrong. That is not a description of the
function — it is a *specification*. Vague prompts get vague functions.

In [ ]:
def strip_magics(source: str) -> str:
    """Drop the leading %magic / !shell lines from cell source."""
    raise NotImplementedError("hand the spec above to your AI, then paste here")

Run the test cell again (click it and press Ctrl+Enter). Six greens or keep going.

If your AI reached for a regular expression and it fails the modulo case, that
is the most common wrong answer — `re.sub(r'^%.*$', '', src, flags=re.M)` strips
matching lines *anywhere* in the cell. Tell it the test that failed and let it
try again. Reading the failure back to the model is the loop.

---
## 2. `get_cells()` — three ways to see the notebook

Now the interesting one. You want the list of cells in this notebook. There is
no function for that, because the kernel does not have the notebook. So you go
and get it — and where from depends entirely on where you are running.

| where | how | catch |
|---|---|---|
| Google Colab | ask the browser: `google.colab._message` | only exists in Colab |
| JupyterHub, the course server | `jupyter-mcp-cli`, a helper that talks to the server | only if installed |
| anywhere with a saved file | read the `.ipynb` off disk | only as fresh as the last save |

None of the three works everywhere. So you write all three and try them in
order, taking the first that answers. This shape — **a list of sources, tried
in order** — comes back in notebook 01 when one model endpoint is full and you
need the other. Watch for it.

In [ ]:
@check("colab source returns a list")
def _():
    assert isinstance(cells_from_colab(), list)

@check("colab source is empty when not in Colab")
def _():
    assert cells_from_colab() == []

@check("disk source reads a notebook file when there is one")
def _():
    import glob
    if not glob.glob("*.ipynb"):
        # No .ipynb on disk means you are almost certainly in Colab, which
        # never writes one. That is the entire reason cells_from_colab exists.
        return
    assert len(cells_from_disk()) > 5

@check("disk source survives a file that is not there")
def _():
    assert cells_from_disk("no_such_file.ipynb") == []

@check("disk source survives a file that is not JSON")
def _():
    open("broken.ipynb", "w").write("{ not json")
    assert cells_from_disk("broken.ipynb") == []

**Give your AI this, one function at a time:**

> 1. `cells_from_colab() -> list[dict]` — import `_message` from `google.colab`,
>    call `_message.blocking_request("get_ipynb", timeout_sec=5)`, and return
>    `reply["ipynb"]["cells"]`. Return `[]` if anything at all goes wrong,
>    including the import failing.
>
> 2. `cells_from_disk(path: str | None = None) -> list[dict]` — read a notebook
>    file as JSON and return its `"cells"` list. If `path` is None, glob
>    `*.ipynb` in the current directory and use the most recently modified one.
>    Return `[]` on any failure. Standard library only.

Both specs end with the same sentence, and it is the important one. **An empty
list is a normal answer here, not an error.** Sometimes there is no notebook —
you are in a test runner, or a plain Python process. A primitive that raises in
that situation makes every function above it need a `try`.

In [ ]:
def cells_from_colab() -> list:
    """Live cells from the Colab frontend, or [] if not in Colab."""
    raise NotImplementedError


def cells_from_disk(path=None) -> list:
    """Cells read off an .ipynb file, or [] on any failure."""
    raise NotImplementedError

The third source shells out to a command-line helper that is installed on the
course server and probably not on your laptop. It follows the same pattern, so
here it is rather than making you write a third near-identical function:

In [ ]:
import json, subprocess

def cells_from_mcp() -> list:
    """Live cells via the jupyter-mcp-cli helper, or [] if it isn't installed."""
    try:
        path = subprocess.run(["jupyter-mcp-cli", "get_active_notebook"],
                              capture_output=True, text=True, timeout=5).stdout.strip()
        if not path or "error" in path.lower():
            return []
        out = subprocess.run(["jupyter-mcp-cli", "read_notebook_cells",
                              "--arg", f"notebook_path={path}"],
                             capture_output=True, text=True, timeout=5).stdout
        return json.loads(out).get("cells", [])
    except Exception:
        return []


def get_cells() -> list:
    """Every cell of the current notebook. The first source that answers wins."""
    return cells_from_colab() or cells_from_mcp() or cells_from_disk()


print("this notebook has", len(get_cells()), "cells")
print("which source answered?",
      "colab" if cells_from_colab() else "mcp" if cells_from_mcp() else "disk")

That `or` chain is the entire fallback. Three function calls, short-circuiting
on the first non-empty list. You will write a more careful version of the same
idea in notebook 01, where the failures are network timeouts rather than empty
lists and you cannot use `or`.

Two small readers now — one-liners, so read them rather than building them:

In [ ]:
def cell_source(cell: dict) -> str:
    """The text of one cell. nbformat allows a string or a list of lines."""
    return "".join(cell.get("source", []))


def get_cell(i: int) -> str:
    """Source of cell i, or "" if there is no cell i."""
    cells = get_cells()
    return cell_source(cells[i]) if -len(cells) <= i < len(cells) else ""


print(repr(get_cell(0)[:60]))
print(repr(get_cell(9999)))     # out of range is "", not an exception

`get_cell(9999)` returning `""` rather than raising is a deliberate choice, and
the same one as the empty list above. Your tool runs inside somebody else's
work. Crashing their kernel because a cell moved is a worse outcome than saying
nothing.

---
## 3. `get_current_cell()` — where am I?

You would think this needs the notebook file. It does not, and the shortcut is
worth knowing.

IPython keeps every cell you have run in `_ih` (input history), and it appends
the source **before** running it. So while your cell is executing, `_ih[-1]`
*is* your cell. Exact, instant, no file involved.

In [ ]:
@check("returns the source of the running cell")
def _():
    assert "def _()" in get_current_cell()

@check("returns a string even with no kernel")
def _():
    assert isinstance(get_current_cell(), str)

**Give your AI this:**

> `get_current_cell() -> str` — return the source of the cell currently
> executing, by reading `shell().user_ns["_ih"][-1]`. Return `""` if `shell()`
> is None or the history is empty.

In [ ]:
def get_current_cell() -> str:
    """Source of the cell that is executing right now, or ""."""
    raise NotImplementedError

---
## 4. `current_index()` — the hard one

`get_current_cell()` gives you the text. It does not give you the **position**,
and position is what you need for anything that says "the cell above me" or
"the cell below me" — which is most of the useful tools.

To find the position you have to match your source against the notebook's cell
list. And here is where `strip_magics` earns its place: the notebook has
`%%explain\nx = 1`, IPython gave you `x = 1`, so you strip both sides before
comparing.

There is a decision buried in this function. What do you do when two cells have
identical source? You cannot tell which one you are in. The tempting answer is
"return the first". The right answer is **return None**, because the caller is
about to *write* to that index, and writing to the wrong cell destroys work
that is not yours.

In [ ]:
FAKE = [
    {"cell_type": "markdown", "source": "# Task"},
    {"cell_type": "code",     "source": "%%explain\nx = 1"},
    {"cell_type": "code",     "source": "y = 2"},
]

@check("finds the cell through its magic line")
def _():
    assert index_of(FAKE, "x = 1") == 1

@check("skips markdown cells")
def _():
    assert index_of([{"cell_type": "markdown", "source": "y = 2"}] + FAKE, "y = 2") == 3

@check("returns None when the source is not there")
def _():
    assert index_of(FAKE, "z = 3") is None

@check("returns None rather than guessing between duplicates")
def _():
    dupes = [{"cell_type": "code", "source": "x = 1"},
             {"cell_type": "code", "source": "x = 1"}]
    assert index_of(dupes, "x = 1") is None

**Give your AI this:**

> `index_of(cells: list[dict], source: str) -> int | None`. Compare
> `strip_magics(source)` against `strip_magics(cell_source(c))` for every cell
> in `cells` whose `cell_type` is `"code"`. If exactly one matches, return its
> index. If none match, or more than one matches, return `None`. Do not use
> fuzzy or approximate matching.

That last sentence is there because your AI will otherwise offer you a
similarity score, and a similarity score here is worse than useless — it turns
"I don't know" into a confident wrong index.

In [ ]:
def index_of(cells: list, source: str):
    """Index of the code cell whose source matches, or None if unsure."""
    raise NotImplementedError


def current_index():
    """Where am I in the notebook? None if unknown or ambiguous."""
    return index_of(get_cells(), get_current_cell())

In [ ]:
print("I am cell", current_index(), "of", len(get_cells()))

def get_cells_before(n: int = 1) -> list:
    """Source of the n cells immediately above this one."""
    i = current_index()
    return [] if i is None else [cell_source(c) for c in get_cells()[max(0, i - n):i]]

print("above me:", [s[:40] for s in get_cells_before(1)])

`get_cells_before` is four lines and no new ideas — it is `get_cells()` and
`current_index()` stuck together. That is worth noticing. From here on, most of
what you want is a **combination** of primitives rather than a new one. Adding a
primitive is expensive; combining two is free.

---
## 5. `format_error(exc)` — errors in a shape a model can use

When a cell fails you get a traceback: twenty lines of frames, file paths, and
arrows. Almost none of it helps a language model, and all of it costs you
context window — the space the student's actual code needs to occupy.

What you want is the last line. `TypeError: unsupported operand type(s) for +`.
Type and message, nothing else.

In [ ]:
@check("gives type and message on one line")
def _():
    try:
        int("banana")
    except ValueError as e:
        assert format_error(e) == "ValueError: invalid literal for int() with base 10: 'banana'"

@check("has no stack frames in it")
def _():
    def inner():
        raise KeyError("k")
    try:
        inner()
    except KeyError as e:
        out = format_error(e)
    assert "Traceback" not in out and "inner" not in out

@check("returns empty string for None")
def _():
    assert format_error(None) == ""

**Give your AI this:**

> `format_error(exc: BaseException | None) -> str` — return the exception type
> and message on a single line, with no stack frames and no trailing newline.
> Use `traceback.format_exception_only`. Return `""` when `exc` is None.

Note the signature takes the **exception**, not the cell result it came from.
The real `socratic-watchdog` has this logic written twice, because one copy
reads it off one kind of object and one off another. Taking the exception
itself as the argument means one function instead of two.

In [ ]:
import traceback

def format_error(exc) -> str:
    """One-line "TypeError: ..." for an exception, or "" for None."""
    raise NotImplementedError

---
# L1 — writing

## 6. `insert_cell_below(source)`

Here is the constraint that shapes every tool in this course, so read it twice.

**The kernel cannot write to the notebook.** It cannot reach into the document.
What it *can* do is attach a small message to its reply and ask the browser
nicely: *"here is some text, please put it in a cell."* That message is called
a `set_next_input` payload, and it can address exactly two places:

- the cell that is running right now (`replace=True`), or
- a new cell just below it (`replace=False`).

That is the entire write API. There is no `write_cell(7, ...)`. Writing to an
arbitrary cell index needs a completely different mechanism — talking to the
Jupyter *server* over HTTP with an authentication token — which is a much
bigger build and breaks in more ways on a shared server. Every tool in this
series is designed to need only these two.

In [ ]:
def insert_cell_below(source: str) -> bool:
    """Ask the frontend to add a new cell below this one."""
    ip = shell()
    if ip is None:
        return False
    ip.set_next_input(source, replace=False)
    return True


def replace_current_cell(source: str) -> bool:
    """Ask the frontend to overwrite the cell that is running."""
    ip = shell()
    if ip is None:
        return False
    ip.set_next_input(source, replace=True)
    return True


print("write primitives ready")

In [ ]:
# Run this in a browser and a new cell appears below, already written.
insert_cell_below("# nbkit wrote this cell\nprint('a tool just edited your notebook')")

If nothing appeared, you are running this notebook headlessly — there is no
browser to receive the payload, so it goes nowhere. That is not a bug you can
fix from the kernel side, and it is worth understanding now rather than
debugging later.

Two things about `replace=True` that will bite you if you skip them:

1. **It is destructive.** The student's code is gone. Their editor's undo still
   works, but yours doesn't exist.
2. **You cannot read back what you wrote.** The frontend applies the payload
   *after* your cell finishes. Inside the same run, `get_current_cell()` still
   returns the old text. Any tool that writes and then re-reads in one pass is
   reading stale data.

## 7. Showing things

The last two are one line each. `IPython.display` already does the work.

In [ ]:
from IPython.display import Markdown, HTML, display

def show_md(text: str) -> None:
    """Render Markdown in the cell output."""
    display(Markdown(text))


def show_html(html: str) -> None:
    """Render raw HTML in the cell output."""
    display(HTML(html))


def live_display(initial: str = ""):
    """An output area you can rewrite in place. handle.update(HTML(...))."""
    return display(HTML(initial), display_id=True)


show_md("**This is Markdown**, rendered from a string — which is how every tool "
        "in this series shows its answer.")

`live_display` is the one worth remembering. It returns a *handle*, and calling
`handle.update(...)` replaces what is on screen instead of printing underneath
it. That is how you show "thinking…" and then swap it for the answer, rather
than leaving a trail of dead status messages down the notebook.

In [ ]:
import time
handle = live_display("<i>thinking…</i>")
time.sleep(1)
handle.update(HTML("<b>…done.</b> Same output area, rewritten."))

---
# What you just built

| | | |
|---|---|---|
| `shell()` | the kernel handle | everything else needs it |
| `strip_magics()` | normalise cell source | so a cell can match itself |
| `get_cells()` | the notebook, three ways | Colab / server / disk |
| `get_cell(i)`, `cell_source()` | one cell | out of range is `""` |
| `get_current_cell()` | the running cell's text | `_ih[-1]`, no file needed |
| `current_index()` | the running cell's position | `None` when unsure |
| `get_cells_before(n)` | context above | a combination, not a primitive |
| `format_error(exc)` | error in one line | frames cost context |
| `insert_cell_below()` | write below | the safe write |
| `replace_current_cell()` | overwrite | destructive, no read-back |
| `show_md`, `show_html`, `live_display` | output | `live_display` rewrites in place |

Eleven functions and none of them is longer than about eight lines. That is the
whole of L0 and L1, and it is the entire vocabulary the rest of this series
uses.

## Before you move on

Compare what you built against `../nbkit.py`, which is the same set written out
properly with the reasoning in the docstrings. Where yours differs, work out
which is right — sometimes it will be yours.

From notebook 01 onward we `import nbkit` instead of retyping these. You have
built them once; that was the point.

## A note on the spec-and-test loop

You just did the same four steps eleven times: read a spec, watch a test fail,
hand the spec to a model, watch the test pass. That loop is the deliverable of
this course, not the functions.

It works on a small model because each step asks for something a small model
can actually do. It also works on a large one, and the reason to keep using it
there is different: the test is what tells you the answer is right. Without it
you are reading code and nodding, which is not verification.